# Session 11 — Hybrids: LLM-based agent plus RL-based agent

**DAT 5668 · Module 6 · Concept session (about 80 minutes)**

**Families this session: both.** The LLM-based agent reads the exception. The RL-based agent, or a rule trained the same way, releases the frequent order.

![Hybrid of an LLM-based agent and an RL-based agent](../../assets/hybrid.png)

## Run of show
- 10 min — Cold call: which box in the Session 1 loop is the Q-table, and which is the tool call?
- 20 min — Three hybrid patterns and who is accountable.
- 35 min — Demo: base-stock executor, email planner, controller that can veto.
- 15 min — Students sketch the same pattern on their Session 6 process. Session 12 scores it.

## Learning goals
- Draw a hybrid where an LLM plans or handles exceptions and a fast policy executes.
- State who is accountable when both systems act.
- Name latency: the email agent must not sit on the critical path of every unit ordered.


## Patterns

1. **LLM proposes, controller validates, fast policy executes.** The usual operations pattern. The daily order stays a rule or an RL policy. The LLM only reads the exception.
2. **RL policy calls an LLM as a tool** when the state is "exception flag on." The policy still owns the action.
3. **LLM as planner over a slow workflow** (research, draft, review) with no RL at all. That is Session 3, not a hybrid.

Accountability: the system that **releases** the order owns the KPI. A planner that "suggested" a bad override does not absorb the stockout if nobody could veto it.


In [ ]:
# Instructional messages. A production agent would read a real inbox through a tool.
MESSAGES = {
    "monday": "",
    "tuesday": "Supplier delay: inbound trailer slips one day.",
    "wednesday": "Promo campaign starts tomorrow on this SKU.",
}


def base_stock(stock, target=2):
    """Fast executor. No language model on this path."""
    return max(0, min(2, target - stock))


def llm_style_planner(message):
    """Exception handler. Returns a proposal, not an order."""
    text = message.lower()
    if "delay" in text:
        return {"propose": "lower_target", "reason": "Inbound will miss today."}
    if "promo" in text:
        return {"propose": "raise_target", "reason": "Campaign will lift demand."}
    return {"propose": "none", "reason": "No exception language."}


def controller(proposal, stock):
    """Person or hard rule. This is the accountability point."""
    if proposal["propose"] == "raise_target" and stock >= 4:
        return {"accept": False, "target": 2, "reason": "Already at capacity."}
    if proposal["propose"] == "raise_target":
        return {"accept": True, "target": 3, "reason": proposal["reason"]}
    if proposal["propose"] == "lower_target":
        return {"accept": True, "target": 1, "reason": proposal["reason"]}
    return {"accept": True, "target": 2, "reason": "No override."}


print("day    stock  message                         proposal        released_order")
stock = 1
for day, message in MESSAGES.items():
    proposal = llm_style_planner(message)
    decision = controller(proposal, stock)
    order = max(0, min(2, decision["target"] - stock))
    print(f"{day:8} {stock:5}  {message[:30]:30}  {proposal['propose']:14}  {order}")
    # Toy demand of 1 so the class can watch stock move.
    stock = min(4, stock + order - 1)


## Latency and the go-live question

The executor above does not wait for a model. The planner runs when a message arrives. If you put an LLM on every order line, you pay latency and failure risk on the common path, which is the opposite of why you automated it.

In Session 12 you will score an LLM design and an RL design on the **same** managerial scorecard.


## Fill this with me

The Tuesday row is a supplier delay and stock is 1. What order should release? Check it against the table you just printed.


In [ ]:
# FILL WITH THE CLASS. An integer order.
tuesday_order = None
print("You said", tuesday_order, "| the Tuesday row above is the answer")
print("Who released it: the email agent, or the fast executor after the veto?")
